# Milestone 1R — audit riproducibile degli output NMF

Questo notebook documenta e richiama l'audit degli output già prodotti dal classificatore TF-IDF + NMF. La logica riutilizzabile è salvata in `src/news_topic_audit.py`; configurazione, test e runner ufficiale sono rispettivamente in `config/topic_audit.json`, `tests/test_news_topic_audit.py` e `scripts/run_topic_audit.py`.

L'audit controlla struttura, pesi, distribuzione dei topic, confidenza normalizzata, fonti, andamento temporale e possibili duplicazioni. Non riaddestra il modello, non assegna etichette semantiche e non seleziona macrotemi.

## Metodo e limiti

- **TF-IDF** rappresenta quanto un termine caratterizza un articolo rispetto al corpus; produce una matrice articolo–termine, ma non comprende il significato politico.
- **NMF** scompone quella matrice in pesi articolo–topic e topic–termine; risponde a quali configurazioni lessicali ricorrono insieme, ma i topic restano microtemi emergenti da interpretare.
- **Confidenza normalizzata** è il peso massimo diviso per la somma dei pesi dell'articolo; aiuta a trovare assegnazioni più o meno nette, ma non è una probabilità calibrata.
- **Duplicati e domini prevalenti** aiutano a individuare possibili effetti di agenzie, testate o boilerplate; non dimostrano da soli che un topic sia spurio.

Il corpus Media Cloud misura come partiti e leader sono raccontati negli articoli raccolti dalle query. Non misura automaticamente comunicazione diretta, comportamento istituzionale, consenso, sentiment verso un partito o stance su una policy.

## Riproduzione ufficiale

Prima del run reale, dalla root del repository eseguire i test:

```bash
python -m unittest tests/test_news_topic_audit.py -v
```

Il comando ufficiale che genera gli output persistenti è:

```bash
python scripts/run_topic_audit.py --config config/topic_audit.json
```

Le celle successive effettuano la stessa chiamata tramite le funzioni salvate in `src/`, per consentire l'esecuzione completa del notebook senza duplicare la logica.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / '.git').exists():
    ROOT = ROOT.parent
if not (ROOT / '.git').exists():
    raise FileNotFoundError('Root Git non trovata dalla directory corrente.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
AUDIT_DIR = ROOT / 'reports' / 'topic_audit'
audit_manifest = json.loads((AUDIT_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
topic_distribution = pd.read_csv(AUDIT_DIR / 'topic_distribution.csv')

print(f"Articoli verificati: {int(topic_distribution['articles'].sum()):,}")
print(f"Topic osservati: {len(topic_distribution)}")
if audit_manifest.get('warnings'):
    print('Avvisi automatici:')
    for warning in audit_manifest['warnings']:
        print(f"- {warning}")

## Come leggere gli output

Aprire `reports/topic_audit/audit_report.md` come indice dell'audit. Le tabelle CSV e il JSON contengono i fatti quantitativi; `run_manifest.json` registra input, hash, configurazione, seed, versioni software e output.

I conteggi e gli avvisi non autorizzano ancora etichette politiche o aggregazioni in macrotemi. La revisione semantica degli articoli rappresentativi appartiene alla Milestone 2 e non viene svolta qui.

# Milestone 2 — campione riproducibile per lo human check

Lo human check serve a distinguere topic politicamente coerenti, topic misti, boilerplate o formati editoriali, contenuti non politici e casi dubbi. La selezione è automatica e riproducibile; la classificazione finale è una decisione umana.

Il runner `scripts/run_topic_human_review.py` applica le regole salvate in `src/topic_human_review.py`: per i topic 1, 8 e 9 seleziona tre record ad alto peso e tre casuali con seed 42; per gli altri topic seleziona un controllo ad alto peso. Esclude duplicati esatti titolo–estratto e diversifica i domini. Questa sezione legge soltanto gli output persistenti già generati.

In [ ]:
HUMAN_REVIEW_DIR = ROOT / 'reports' / 'topic_human_review'
review_sample = pd.read_csv(HUMAN_REVIEW_DIR / 'review_sample.csv', encoding='utf-8-sig', keep_default_na=False)
selection_summary = pd.read_csv(HUMAN_REVIEW_DIR / 'selection_summary.csv')

print(f'Righe del campione: {len(review_sample)}')
print(f'Topic coperti: {review_sample["topic_id"].nunique()}')
display(selection_summary[['topic_id', 'tipo_selezione', 'records', 'unique_domains']])

In [ ]:
preview_columns = [
    'review_id', 'topic_id', 'tipo_selezione', 'domain', 'title',
    'peso_topic_dominante', 'confidenza_topic', 'valutazione_preliminare'
]
display(review_sample[preview_columns].head(5))

## Compilazione umana

Aprire `reports/topic_human_review/review_sample.csv` e compilare soltanto `classificazione_umana`, `etichetta_tema_proposta`, `boilerplate_si_no`, `duplicato_sospetto_si_no`, `decisione_inclusione` e `note_revisore`. Non modificare `review_id`, provenienza, testo o campi di selezione.

Le categorie e i valori ammessi sono descritti in `reports/topic_human_review/review_guide.md`. Il fatto che un record sia stato scelto come `artifact_check` è un criterio automatico di controllo, non un giudizio già formulato sul suo contenuto. Il campione non misura prevalenza, consenso o stance.